# Spatial context maps for ICB responders and non-responders as well as patients who did or did not progress following initial local disease

Import packages and neighborhood labeled data

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%pylab inline
import seaborn as sns
import networkx as nx
import math
from itertools import product
import scipy as sp

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


C:\ProgramData\Anaconda3\envs\spacec_new\lib\site-packages\IPython\core\magics\pylab.py:162: UserWarning: pylab import has clobbered these variables: ['product']
`%matplotlib` prevents importing * from pylab and numpy
  warn("pylab import has clobbered these variables: %s"  % clobbered +


In [9]:
from sklearn.neighbors import NearestNeighbors
import time
import os
from itertools import product
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from sklearn.cluster import MiniBatchKMeans
import scanpy as sc

def get_windows(job,n_neighbors):
    start_time,idx,tissue_name,indices = job
    job_start = time.time()
    
    print ("Starting:", str(idx+1)+'/'+str(len(exps)),': ' + exps[idx])

    tissue = tissue_group.get_group(tissue_name)
    to_fit = tissue.loc[indices][[X,Y]].values

    fit = NearestNeighbors(n_neighbors=n_neighbors).fit(tissue[[X,Y]].values)
    m = fit.kneighbors(to_fit)

    #sort_neighbors
    args = m[0].argsort(axis = 1)
    add = np.arange(m[1].shape[0])*m[1].shape[1]
    sorted_indices = m[1].flatten()[args+add[:,None]]

    neighbors = tissue.index.values[sorted_indices]
   
    end_time = time.time()
   
    print ("Finishing:", str(idx+1)+"/"+str(len(exps)),": "+ exps[idx],end_time-job_start,end_time-start_time)
    return neighbors.astype(np.int32)

In [10]:
path_to_data = 'Z:/Max/MCC_project/CODEX_final_Data/CODEX_Annotations/Submission/MCC_single_cell_neighborhoods_markers.csv'
df = pd.read_csv(path_to_data)

Import modules from cellhier for neighborhoods

In [ ]:
import sys
from cellhier.knn_graph_neighborhood import Neighborhoods

## Create cell neighborhoods with a window size of 100 nearest neighbors with regard to the neighborhood
### Parameters to define: window_size = 100
### Threshold for CNs within a given window: 0.85

In [ ]:
import os
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# ==============================================================================
# 1. SETUP INITIAL STATIC METADATA VARIABLES & PALETTE
# ==============================================================================
cluster_col = 'CN_k20_n25_annot'
sum_cols = df[cluster_col].unique()
keep_cols = [
    'UID', "unique_core_num", "cell_types_mediumres",
    'CN_k20_n25_annot', 'CPI_ORR', 'CPI_Progress_status', "Progress_to_metastatic_disease", 
    "Local_disease", 'Baseline_FU_biopsy', 'x', 'y'
]
X = 'x'
Y = 'y'
Reg = 'unique_core_num'

# Global column export tracker requested for CSV files
columns_to_export = [
    'combination', 'combination_num', 'unique_core_num', 'UID', 
    'CN_k20_n25_annot', 'Baseline_FU_biopsy', 'Local_disease', 
    'Progress_to_metastatic_disease', 'CPI_ORR', 'CPI_Progress_status'
]

pal_color = {
    'Tumor boundary': '#5d5c61',
    'Myeloid-cell enriched immune infiltrate': 'sienna',
    'CAF-enriched stroma': 'orange',
    'B cell priming zone': '#3198b5',
    'T cell priming and activation': '#44CB63',
    'Bulk tumor': '#204F89',
    'Vessel-enriched stroma': 'red',
    'Plasma-cell enriched immune infiltrate': 'purple',
    'Epithelium': 'Magenta',
    'Follicle': '#0762eb',
    'Granulocyte-enriched': 'cyan'
}
l = list(pal_color.keys())
palt = pal_color
os.makedirs(save_path, exist_ok=True)

# ==============================================================================
# 2. CORE HELPER FUNCTIONS
# ==============================================================================
def get_thresh_simps(x, thresh):
    sorts = np.argsort(-x, axis=1)
    x_sorted = -np.sort(-x, axis=1)
    cumsums = np.cumsum(x_sorted, axis=1)
    thresh_simps = pd.Series([
        tuple(sorted(sorts[i, :(1 + j)])) 
        for i, j in enumerate(np.argmax(cumsums > thresh, axis=1))
    ])
    return thresh_simps

# --- Refactored Function 1: CPI Response Subset Network & CSV Export ---
def get_network_cpi(subset_list, ttl_per_thres, comb_per_thres, current_windows, current_n, fig_size=(10, 12), save_name=None):
    w_sub_baseline = current_windows[current_n].copy()
    
    # CRITICAL FILTER: Restrict to baseline tissues ('pre', 'FU') and cohort subset
    w_sub_baseline = w_sub_baseline[
        w_sub_baseline["Baseline_FU_biopsy"].isin(["pre", "FU"]) & 
        w_sub_baseline["CPI_ORR"].isin(subset_list)
    ]
    
    if len(w_sub_baseline) == 0:
        print(f"Skipping CPI network {save_name}: No data matching subset criteria.")
        return
        
    xm = w_sub_baseline.loc[:, l].values / current_n
    simps = get_thresh_simps(xm, ttl_per_thres)
    
    # Map spatial context combinations back onto metadata frame for CSV export
    w_sub_baseline['combination'] = [tuple(l[a] for a in s) for s in simps]
    w_sub_baseline['combination_num'] = [tuple(a for a in s) for s in simps]
    
    # Export full subset tracking frame 
    if save_name is not None:
        csv_filename = f"Full_DataFrame_{save_name}.csv"
        w_sub_baseline[columns_to_export].to_csv(os.path.join(save_path, csv_filename), index=False)
    
    simp_freqs = simps.value_counts(normalize=True)
    g = nx.DiGraph()
    selected_simps = simp_freqs[simp_freqs >= comb_per_thres].index.values
          
    for e0 in selected_simps:
        for e1 in selected_simps:
            if (set(list(e0)) < set(list(e1))) and (len(e1) == len(e0) + 1):
                g.add_edge(e0, e1)   

    if len(g.nodes) == 0:
        return

    draw = g
    pos = nx.drawing.nx_pydot.graphviz_layout(draw, prog='dot')
    delta = 8

    plt.figure(figsize=fig_size)
    for n in draw.nodes():
        col = 'black'
        freq_idx = list(simp_freqs.index).index(n)
        plt.scatter(pos[n][0], pos[n][1] - 5, s=simp_freqs.iloc[freq_idx] * 10000, c=col, zorder=-1)
        plt.scatter([pos[n][0]] * len(n), [pos[n][1] + delta * (i + 1) for i in range(len(n))], c=[palt[l[i]] for i in n], marker='s', zorder=5, s=400)

    for e0, e1 in draw.edges():
        plt.plot([pos[e0][0], pos[e1][0]], [pos[e0][1], pos[e1][1]], color='black', linewidth=0.2, alpha=0.3, zorder=-10)

    # Generate Legends (Scaled down to fit fig_size=(10,12) safely)
    freqs = simp_freqs * 10000
    max_size = max(freqs) if len(freqs) > 0 else 10000
    sizes = [round(max_size) / 4, round(max_size) / 2, round(max_size)]
    labels = [
        str(round(max_size / 100) / 4) + "%",
        str(round(max_size / 100) / 2) + "%",
        str(round(max_size / 100)) + "%",
    ]

    legend_elements = [
        plt.Line2D(
            [0], [0],
            marker="o", color="w", label=label,
            markerfacecolor="black", markersize=size**0.5
        )
        for size, label in zip(sizes, labels)
    ]

    legend1 = plt.legend(
        handles=legend_elements,
        loc="lower right",
        title="Total frequency",
        title_fontsize=12, fontsize=10, handlelength=3, handletextpad=1,
        bbox_to_anchor=(0.0, -0.2, 1.0, 0.102),
    )

    legend_patches = [
        mpatches.Patch(color=color, label=label) for label, color in palt.items()
    ]

    plt.gca().add_artist(legend1)
    plt.legend(
        handles=legend_patches,
        bbox_to_anchor=(0.0, -0.2, 1.0, 0.102),
        loc="lower left",
        ncol=2, borderaxespad=0.0, fontsize=10,
    )

    plt.axis('off')
    if save_name is not None:
        plt.savefig(os.path.join(save_path, f"{save_name}.pdf"), bbox_inches='tight')
    plt.close()

# --- Refactored Function 2: Metastatic Progression Subset Network & CSV Export ---
def get_network_progression(subset_list, ttl_per_thres, comb_per_thres, current_windows, current_n, fig_size=(10, 12), save_name=None):
    w_local_disease_baseline = current_windows[current_n].copy()
    
    # FILTER: Restricts strictly to baseline initial local disease profiles
    w_local_disease_baseline = w_local_disease_baseline[
        w_local_disease_baseline["Baseline_FU_biopsy"].isin(["pre"]) & 
        w_local_disease_baseline["Local_disease"].isin(["1"]) & 
        w_local_disease_baseline["Progress_to_metastatic_disease"].isin(subset_list)
    ]
    
    if len(w_local_disease_baseline) == 0:
        print(f"Skipping Progression network {save_name}: No data matching subset criteria.")
        return
        
    xm = w_local_disease_baseline.loc[:, l].values / current_n
    simps = get_thresh_simps(xm, ttl_per_thres)
    
    # Map spatial context combinations back onto metadata frame for CSV export
    w_local_disease_baseline['combination'] = [tuple(l[a] for a in s) for s in simps]
    w_local_disease_baseline['combination_num'] = [tuple(a for a in s) for s in simps]
    
    # Export full subset tracking frame 
    if save_name is not None:
        csv_filename = f"Full_DataFrame_{save_name}.csv"
        w_local_disease_baseline[columns_to_export].to_csv(os.path.join(save_path, csv_filename), index=False)
        
    simp_freqs = simps.value_counts(normalize=True)
    g = nx.DiGraph()
    selected_simps = simp_freqs[simp_freqs >= comb_per_thres].index.values
          
    for e0 in selected_simps:
        for e1 in selected_simps:
            if (set(list(e0)) < set(list(e1))) and (len(e1) == len(e0) + 1):
                g.add_edge(e0, e1)   

    if len(g.nodes) == 0:
        return

    draw = g
    pos = nx.drawing.nx_pydot.graphviz_layout(draw, prog='dot')
    delta = 8

    plt.figure(figsize=fig_size)
    for n in draw.nodes():
        col = 'black'
        freq_idx = list(simp_freqs.index).index(n)
        plt.scatter(pos[n][0], pos[n][1] - 5, s=simp_freqs.iloc[freq_idx] * 10000, c=col, zorder=-1)
        plt.scatter([pos[n][0]] * len(n), [pos[n][1] + delta * (i + 1) for i in range(len(n))], c=[palt[l[i]] for i in n], marker='s', zorder=5, s=400)

    for e0, e1 in draw.edges():
        plt.plot([pos[e0][0], pos[e1][0]], [pos[e0][1], pos[e1][1]], color='black', linewidth=0.2, alpha=0.3, zorder=-10)

    # Generate Legends (Scaled down to fit fig_size=(10,12) safely)
    freqs = simp_freqs * 10000
    max_size = max(freqs) if len(freqs) > 0 else 10000
    sizes = [round(max_size) / 4, round(max_size) / 2, round(max_size)]
    labels = [
        str(round(max_size / 100) / 4) + "%",
        str(round(max_size / 100) / 2) + "%",
        str(round(max_size / 100)) + "%",
    ]

    legend_elements = [
        plt.Line2D(
            [0], [0],
            marker="o", color="w", label=label,
            markerfacecolor="black", markersize=size**0.5
        )
        for size, label in zip(sizes, labels)
    ]

    legend1 = plt.legend(
        handles=legend_elements,
        loc="lower right",
        title="Total frequency",
        title_fontsize=12, fontsize=10, handlelength=3, handletextpad=1,
        bbox_to_anchor=(0.0, -0.2, 1.0, 0.102),
    )

    legend_patches = [
        mpatches.Patch(color=color, label=label) for label, color in palt.items()
    ]

    plt.gca().add_artist(legend1)
    plt.legend(
        handles=legend_patches,
        bbox_to_anchor=(0.0, -0.2, 1.0, 0.102),
        loc="lower left",
        ncol=2, borderaxespad=0.0, fontsize=10,
    )

    plt.axis('off')
    if save_name is not None:
        plt.savefig(os.path.join(save_path, f"{save_name}.pdf"), bbox_inches='tight')
    plt.show()

# ==============================================================================
# 3. AUTOMATED SENSITIVITY SWEEP PARAMETER LOOP
# ==============================================================================
window_sizes = [100]
threshold_values = [0.85]

for n_num in window_sizes:
    print(f"--- Computing Windows, Graphs & CSV Exports for Window Size: n = {n_num} ---")
    
    ks = [n_num]
    Neigh = Neighborhoods(df, ks, cluster_col, sum_cols, keep_cols, X, Y, reg=Reg, add_dummies=True)
    windows = Neigh.k_windows()
    
    for per_keep_thres in threshold_values:
        thresh_str = f"t{int(per_keep_thres * 100)}"
        
        # A. CPI Response Cohort Sweep + Exports
        name_cpi_resp = f"CPI_Response_n{n_num}_{thresh_str}"
        get_network_cpi(
            subset_list=['1'], ttl_per_thres=per_keep_thres, comb_per_thres=0.007, 
            current_windows=windows, current_n=n_num, fig_size=(10, 12), save_name=name_cpi_resp
        )
        
        name_cpi_nonresp = f"CPI_NonResponse_n{n_num}_{thresh_str}"
        get_network_cpi(
            subset_list=['0'], ttl_per_thres=per_keep_thres, comb_per_thres=0.007, 
            current_windows=windows, current_n=n_num, fig_size=(10, 12), save_name=name_cpi_nonresp
        )
        
        # B. Local Disease Metastatic Progression Sweep + Exports
        name_prog = f"Progression_DistantMetastasis_n{n_num}_{thresh_str}"
        get_network_progression(
            subset_list=['1'], ttl_per_thres=per_keep_thres, comb_per_thres=0.007, 
            current_windows=windows, current_n=n_num, fig_size=(10, 12), save_name=name_prog
        )
        
        name_no_prog = f"No_Progression_LocalDisease_n{n_num}_{thresh_str}"
        get_network_progression(
            subset_list=['0'], ttl_per_thres=per_keep_thres, comb_per_thres=0.007, 
            current_windows=windows, current_n=n_num, fig_size=(10, 12), save_name=name_no_prog
        )

print("Sensitivity Analysis execution completed. Both plots (.pdf) and raw segment matrices (.csv) have been saved.")

--- Computing Windows, Graphs & CSV Exports for Window Size: n = 100 ---
Starting: 70/356 : TMA_1_A1
Finishing: 70/356 : TMA_1_A1 1.8014192581176758 1.8653979301452637
Starting: 58/356 : TMA_1_A2
Finishing: 58/356 : TMA_1_A2 0.6307945251464844 2.5221850872039795
Starting: 44/356 : TMA_1_A3
Finishing: 44/356 : TMA_1_A3 0.20893216133117676 2.7481143474578857
Starting: 32/356 : TMA_1_A4
Finishing: 32/356 : TMA_1_A4 0.86273193359375 3.6178436279296875
Starting: 20/356 : TMA_1_A5
Finishing: 20/356 : TMA_1_A5 0.6767802238464355 4.318606615066528
Starting: 76/356 : TMA_1_B1
Finishing: 76/356 : TMA_1_B1 0.685779333114624 5.024381160736084
Starting: 63/356 : TMA_1_B2
Finishing: 63/356 : TMA_1_B2 0.29790472984313965 5.341278791427612
Starting: 50/356 : TMA_1_B3
Finishing: 50/356 : TMA_1_B3 0.46284961700439453 5.813125371932983
Starting: 37/356 : TMA_1_B4
Finishing: 37/356 : TMA_1_B4 0.3238973617553711 6.153016805648804
Starting: 25/356 : TMA_1_B5
Finishing: 25/356 : TMA_1_B5 0.308901309967041 6.

Finishing: 101/356 : TMA_22_A5 0.4438588619232178 52.54006361961365
Starting: 124/356 : TMA_22_B1
Finishing: 124/356 : TMA_22_B1 0.731762170791626 53.28582143783569
Starting: 116/356 : TMA_22_B2
Finishing: 116/356 : TMA_22_B2 0.6297991275787354 53.9346137046814
Starting: 165/356 : TMA_22_B3
Finishing: 165/356 : TMA_22_B3 0.9127073287963867 54.864314556121826
Starting: 140/356 : TMA_22_B4
Finishing: 140/356 : TMA_22_B4 0.06697845458984375 54.95728325843811
Starting: 132/356 : TMA_22_B5
Finishing: 132/356 : TMA_22_B5 0.5668177604675293 55.52709984779358
Starting: 138/356 : TMA_22_B6
Finishing: 138/356 : TMA_22_B6 0.6577863693237305 56.1998815536499
Starting: 94/356 : TMA_22_B7
Finishing: 94/356 : TMA_22_B7 0.6737856864929199 56.89366054534912
Starting: 146/356 : TMA_22_C1
Finishing: 146/356 : TMA_22_C1 0.21093273162841797 57.122586488723755
Starting: 117/356 : TMA_22_C2
Finishing: 117/356 : TMA_22_C2 0.479846715927124 57.609429121017456
Starting: 143/356 : TMA_22_C3
Finishing: 143/356 : 

Finishing: 224/356 : TMA_3_A2 0.5048379898071289 100.16671061515808
Starting: 212/356 : TMA_3_A3
Finishing: 212/356 : TMA_3_A3 0.22992396354675293 100.41263008117676
Starting: 199/356 : TMA_3_A4
Finishing: 199/356 : TMA_3_A4 0.7437596321105957 101.16238927841187
Starting: 187/356 : TMA_3_A5
Finishing: 187/356 : TMA_3_A5 0.7917444705963135 101.97412657737732
Starting: 240/356 : TMA_3_B1
Finishing: 240/356 : TMA_3_B1 0.5168330669403076 102.5109531879425
Starting: 228/356 : TMA_3_B2
Finishing: 228/356 : TMA_3_B2 0.6128013134002686 103.14075112342834
Starting: 217/356 : TMA_3_B3
Finishing: 217/356 : TMA_3_B3 0.407867431640625 103.56461381912231
Starting: 204/356 : TMA_3_B4
Finishing: 204/356 : TMA_3_B4 0.2269277572631836 103.80353784561157
Starting: 191/356 : TMA_3_B5
Finishing: 191/356 : TMA_3_B5 0.23592329025268555 104.0454592704773
Starting: 180/356 : TMA_3_B6
Finishing: 180/356 : TMA_3_B6 0.6317963600158691 104.68625330924988
Starting: 173/356 : TMA_3_B7
Finishing: 173/356 : TMA_3_B7 0

Finishing: 271/356 : TMA_41_B7 0.36388468742370605 152.07097959518433
Starting: 279/356 : TMA_41_C2
Finishing: 279/356 : TMA_41_C2 0.4228630065917969 152.5038390159607
Starting: 278/356 : TMA_41_C4
Finishing: 278/356 : TMA_41_C4 0.8357319831848145 153.3525652885437
Starting: 277/356 : TMA_41_C5
Finishing: 277/356 : TMA_41_C5 0.9017078876495361 154.2782654762268
Starting: 276/356 : TMA_41_C7
Finishing: 276/356 : TMA_41_C7 0.3338935375213623 154.63515162467957
Starting: 285/356 : TMA_41_D2
Finishing: 285/356 : TMA_41_D2 0.5598294734954834 155.2059781551361
Starting: 284/356 : TMA_41_D3
Finishing: 284/356 : TMA_41_D3 1.0256669521331787 156.24863052368164
Starting: 283/356 : TMA_41_D4
Finishing: 283/356 : TMA_41_D4 1.1426331996917725 157.4182550907135
Starting: 282/356 : TMA_41_D5
Finishing: 282/356 : TMA_41_D5 0.6008045673370361 158.04905009269714
Starting: 281/356 : TMA_41_D6
Finishing: 281/356 : TMA_41_D6 0.5308310985565186 158.59587621688843
Starting: 280/356 : TMA_41_D7
Finishing: 280

Starting: 325/356 : TMA_5_4C
Finishing: 325/356 : TMA_5_4C 1.0576584339141846 204.57305908203125
Starting: 323/356 : TMA_5_4D
Finishing: 323/356 : TMA_5_4D 0.9936800003051758 205.5947299003601
Starting: 322/356 : TMA_5_5A
Finishing: 322/356 : TMA_5_5A 0.31189727783203125 205.93461775779724
Starting: 326/356 : TMA_5_5B
Finishing: 326/356 : TMA_5_5B 0.4638516902923584 206.40846753120422
Starting: 329/356 : TMA_6_A_3
Finishing: 329/356 : TMA_6_A_3 0.13095784187316895 206.5534212589264
Starting: 328/356 : TMA_6_A_4
Finishing: 328/356 : TMA_6_A_4 0.11496162414550781 206.67338156700134
Starting: 327/356 : TMA_6_A_5
Finishing: 327/356 : TMA_6_A_5 0.0659785270690918 206.7423596382141
Starting: 332/356 : TMA_6_A_6
Finishing: 332/356 : TMA_6_A_6 0.039988040924072266 206.78534626960754
Starting: 331/356 : TMA_6_A_7
Finishing: 331/356 : TMA_6_A_7 0.022992849349975586 206.81133770942688
Starting: 330/356 : TMA_6_A_8
Finishing: 330/356 : TMA_6_A_8 0.012997627258300781 206.8263349533081
Starting: 333

-------------------------